# Cross-dataset foundation-model benchmark

This notebook will compare frozen foundation-model representations across:

1. ARCHS4 unseen samples from studies represented during training
2. ARCHS4 unseen samples from completely unseen studies
3. GTEx
4. TCGA
5. OSDR (external mouse data; added in a later section)

The first section loads and validates the two ARCHS4 controls. These controls separate **sample novelty** from **study novelty** before external dataset shift is introduced. The notebook uses precomputed ARCHS4 embeddings and never reruns the foundation model for these samples.


## 1. Environment and paths

Run this notebook from anywhere inside the repository. The cell below locates the repository root and imports the reusable cohort module from `src/`.


In [ ]:
from pathlib import Path
import json
import sys

import numpy as np
import pandas as pd

REPO_ROOT = Path.cwd().resolve()
while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / 'src' / 'fm_embed').is_dir():
    REPO_ROOT = REPO_ROOT.parent
if not (REPO_ROOT / 'src' / 'fm_embed').is_dir():
    raise RuntimeError('Could not locate the bridge-rna repository root')

sys.path.insert(0, str(REPO_ROOT / 'src'))

from fm_embed.cohorts import available_cohorts, load_archs4_cohort

pd.set_option('display.max_columns', 100)
print('Repository:', REPO_ROOT)
print('Available cohorts:')
for name, purpose in available_cohorts().items():
    print(f'  {name}: {purpose}')


## 2. Load the two ARCHS4 benchmark cohorts

- `unseen_sample_seen_study`: exact GSM was held out, but at least one candidate GSE occurred in training. This measures within-study generalization.
- `strict_unseen_single_gse`: exact GSM was held out, maps to one GSE, and that GSE was absent from training. This is the clean primary study-level benchmark.

Raw ARCHS4 metadata is joined from the versioned v2.5 metadata snapshot. Embeddings remain lazy until `load_embeddings()` or `iter_batches()` is called.


In [ ]:
ARCHS4_METADATA_FIELDS = [
    'series_id',
    'organism_ch1',
    'title',
    'source_name_ch1',
    'characteristics_ch1',
    'platform_id',
    'library_strategy',
    'instrument_model',
    'singlecellprobability',
    'archs4_version',
]

seen_study = load_archs4_cohort(
    'unseen_sample_seen_study',
    include_archs4_metadata=True,
    metadata_columns=ARCHS4_METADATA_FIELDS,
)

unseen_study = load_archs4_cohort(
    'strict_unseen_single_gse',
    include_archs4_metadata=True,
    metadata_columns=ARCHS4_METADATA_FIELDS,
)

print(f'Seen-study holdout: {len(seen_study):,} samples')
print(f'Unseen-study holdout: {len(unseen_study):,} samples')


## 3. Cohort composition

Summaries are calculated from metadata only; no embedding matrix is materialized. Candidate-GSE counts include every preserved candidate for multi-GSE samples.


In [ ]:
def candidate_gses(metadata):
    return {gse for candidates in metadata['gse_candidates'] for gse in candidates}

def summarize_cohort(cohort):
    metadata = cohort.metadata
    species = metadata['species'].value_counts()
    status = metadata['mapping_status'].value_counts()
    return {
        'cohort': cohort.name,
        'samples': len(cohort),
        'human': int(species.get('human', 0)),
        'mouse': int(species.get('mouse', 0)),
        'candidate_GSEs': len(candidate_gses(metadata)),
        'mapped_single': int(status.get('mapped_single', 0)),
        'mapped_multiple': int(status.get('mapped_multiple', 0)),
        'title_coverage': metadata['title'].notna().mean(),
        'embedding_dim': cohort.embedding_dim,
    }

cohort_summary = pd.DataFrame([
    summarize_cohort(seen_study),
    summarize_cohort(unseen_study),
]).set_index('cohort')
cohort_summary


## 4. Leakage and cohort-definition checks

These assertions are part of the benchmark definition. The seen-study holdout must overlap training at the GSE level, while the strict unseen-study cohort must not. Neither cohort may contain a training GSM.


In [ ]:
manifest_path = REPO_ROOT / 'data' / 'manifests' / 'sample_manifest.parquet'
manifest = pd.read_parquet(
    manifest_path,
    columns=['gsm', 'split', 'gse_candidates'],
)
train = manifest[manifest['split'].eq('train')]
train_gsms = set(train['gsm'])
train_gses = candidate_gses(train)
seen_gses = candidate_gses(seen_study.metadata)
unseen_gses = candidate_gses(unseen_study.metadata)

assert set(seen_study.metadata['gsm']).isdisjoint(train_gsms)
assert set(unseen_study.metadata['gsm']).isdisjoint(train_gsms)
assert set(seen_study.metadata['gsm']).isdisjoint(unseen_study.metadata['gsm'])
assert seen_study.metadata['study_exposure'].eq('seen_study').all()
assert unseen_study.metadata['study_exposure'].eq('unseen_study').all()
assert unseen_study.metadata['mapping_status'].eq('mapped_single').all()
assert seen_gses & train_gses
assert not (unseen_gses & train_gses)

qa = pd.Series({
    'GSM overlap between benchmark cohorts': len(
        set(seen_study.metadata['gsm']) & set(unseen_study.metadata['gsm'])
    ),
    'seen-study GSE overlap with training': len(seen_gses & train_gses),
    'strict-unseen GSE overlap with training': len(unseen_gses & train_gses),
    'seen-study missing embedding indices': int(seen_study.metadata['global_index'].isna().sum()),
    'unseen-study missing embedding indices': int(unseen_study.metadata['global_index'].isna().sum()),
})
qa


## 5. Inspect aligned metadata and embeddings

The test below reads only a small batch. For full analyses, prefer `iter_batches()` unless the complete cohort comfortably fits in memory. Each metadata row is aligned with the embedding at the same batch position.


In [ ]:
metadata_batch, embedding_batch = next(
    unseen_study.iter_batches(batch_size=8)
)

assert len(metadata_batch) == len(embedding_batch)
assert embedding_batch.shape == (8, 512)
assert embedding_batch.dtype == np.float32

display(metadata_batch[[
    'gsm', 'species', 'gse_candidates', 'study_exposure',
    'title', 'source_name_ch1', 'platform_id', 'global_index',
]])
print('Embedding batch:', embedding_batch.shape, embedding_batch.dtype)


## 6. Freeze cohort provenance

The ordered GSM checksum makes later results traceable to the exact cohort order. Save these dictionaries alongside figures and metric tables.


In [ ]:
benchmark_provenance = {
    'seen_study': seen_study.provenance(),
    'unseen_study': unseen_study.provenance(),
}
print(json.dumps(benchmark_provenance, indent=2))


## Next sections

The next notebook update will add TPM-based GTEx and TCGA embedding inputs, harmonized sample metadata, and matched cross-dataset benchmark tasks. **CPM preprocessing will not be used.**
